In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

# Q1. Generating questions

In [4]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [5]:
from openai import OpenAI
from dotenv import load_dotenv


client = OpenAI()

load_dotenv()

True

In [6]:
def create_user_prompt(doc):
    return f"""
Filename:
{doc['filename']}

Content:
{doc['content']}
""".strip()

In [7]:
documents_small = documents[:3]
for doc in documents_small:
    print(doc["filename"])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [8]:
from evaluation_utils import llm_structured
results = []
usages = []

for doc in documents_small:

    user_prompt = create_user_prompt(doc)

    questions, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    results.append(questions)
    usages.append(usage)

In [9]:
for usage in usages:
    print(
        usage.input_tokens,
        usage.output_tokens
    )

959 101
1173 87
1585 120


In [10]:
avg_input_tokens = sum(
    usage.input_tokens 
    for usage in usages
) / len(usages)

print(avg_input_tokens)

1239.0


# Q2

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [13]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [14]:
import pandas as pd

df = pd.read_csv("ground-truth.csv")

ground_truth = df.to_dict(orient="records")
q = ground_truth[0]["question"]

In [15]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [16]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)
def text_search(query, num_results=5):

    return index.search(
        query,
        num_results=num_results
    )

results = text_search(q)

In [17]:
results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

# Q3

In [18]:
from embedder import Embedder

embed = Embedder()


X = embed.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [19]:
from minsearch import VectorSearch
vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(
    X,
    chunks
)

In [20]:
def vector_search(query, num_results=5):

    query_vector = embed.encode(query)

    return vindex.search(
        query_vector,
        num_results=num_results
    )

In [21]:
results = vector_search(q)
results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

# Q4

In [22]:
def compute_relevance(results, ground_truth_filename):
    relevance = []

    for doc in results:
        if doc["filename"] == ground_truth_filename:
            relevance.append(1)
        else:
            relevance.append(0)

    return relevance

In [23]:

def hit_rate(relevance):
    return 1 if 1 in relevance else 0

def mrr(relevance):
    for idx, value in enumerate(relevance):
        if value == 1:
            return 1 / (idx + 1)

    return 0

In [24]:


from tqdm.auto import tqdm
def evaluate(ground_truth, search_function):

    hit_rates = []
    mrr_scores = []

    for item in tqdm(ground_truth):

        question = item["question"]
        filename = item["filename"]

        results = search_function(
            question,
            num_results=5
        )

        relevance = compute_relevance(
            results,
            filename
        )

        hit_rates.append(
            hit_rate(relevance)
        )

        mrr_scores.append(
            mrr(relevance)
        )


    return {
        "hit_rate": sum(hit_rates) / len(hit_rates),
        "mrr": sum(mrr_scores) / len(mrr_scores)
    }

In [25]:
text_eval = evaluate(
    ground_truth,
    text_search
)

text_eval

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592593}

# Q5

In [26]:
vector_eval = evaluate(
    ground_truth,
    vector_search
)
vector_eval

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

# Q6

In [31]:
def hybrid_search(query, k=60, num_results=5):

    text_results = text_search(
        query,
        num_results=10
    )

    vector_results = vector_search(
        query,
        num_results=10
    )

    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=num_results
    )

In [32]:
for k in [1,50,100,200]:

    results = evaluate(
        ground_truth,
        lambda q, num_results=5: hybrid_search(
            q,
            k=k,
            num_results=num_results
        )
    )

    print(k, results)

  0%|          | 0/360 [00:00<?, ?it/s]

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444444}


  0%|          | 0/360 [00:00<?, ?it/s]

50 {'hit_rate': 0.8361111111111111, 'mrr': 0.6379166666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

100 {'hit_rate': 0.8361111111111111, 'mrr': 0.6379166666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

200 {'hit_rate': 0.8361111111111111, 'mrr': 0.6379166666666667}


k=1